# SDMAD SST / LST Pipeline — Philippine Area of Responsibility

Temperature **and anomaly** maps over the Philippine Area of Responsibility (PAR) for a **time range**:

| Product | Source | Outputs |
|---|---|---|
| **SST** | JAXA GCOM-C/SGLI L3 SST (daily), Google Earth Engine (`ee-chriscandido93`) | daily + monthly |
| **LST** | MODIS MOD11A2 (Terra) + MYD11A2 (Aqua) 8-day 1 km, Planetary Computer `modis-11A2-061` | monthly |

**Climatology baseline: 2018–2025.** 2026 onwards is never used in the climatology.
Daily SST anomalies compare each day with the mean of a 15-day window centred on that day (± 7 days) in every
baseline year; monthly anomalies compare with the same span in every baseline year.

**How a range is processed.** It is split by calendar month into
```
SDMAD_Products/outputs/<PRODUCT>/<YYYY>/<MM>/daily/     <PRODUCT>_{temperature,anomaly}_<YYYYMMDD>.{tif,png,json}   (SST)
SDMAD_Products/outputs/<PRODUCT>/<YYYY>/<MM>/monthly/   <PRODUCT>_{temperature,anomaly}_<YYYY-MM>.{tif,png,json}
SDMAD_Products/outputs/<PRODUCT>/animations/            <PRODUCT>_{temperature,anomaly}_<start>_<end>.gif
```
Nothing is processed twice: days already written are skipped, and a month's composite is only redone when the range
adds days to it. Days after the newest available image are reported as *pending* and filled in by a later run.

**Before running:** upload this repository to Google Drive as `MyDrive/SDMAD_Products`
(so that `MyDrive/SDMAD_Products/pipeline/` exists). Then run the cells top to bottom.

In [ ]:
#@title 1. Mount Google Drive and load the pipeline
PIPELINE_DIR = "/content/drive/MyDrive/SDMAD_Products"  #@param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")
if not os.path.isdir(os.path.join(PIPELINE_DIR, "pipeline")):
    raise FileNotFoundError(
        f"No 'pipeline' folder in {PIPELINE_DIR}. Upload the repository contents to that folder first.")

# numpy is pinned below 2.3 (requirements.txt) because Colab's preinstalled numba requires it. If you see
# "numba requires numpy<2.3", or numpy was imported before this cell: Runtime > Restart session, then run again.
%pip install -q -r "{PIPELINE_DIR}/requirements.txt" geemap

if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
os.chdir(PIPELINE_DIR)
# Python keeps already-imported code in memory: after replacing the pipeline files in Drive,
# use Runtime > Restart session before running the notebook again.
print("Pipeline directory:", PIPELINE_DIR)

In [ ]:
#@title 2. Inputs
#@markdown ### Time range (inclusive)
START_DATE = "2026-10-01"  #@param {type:"date"}
END_DATE = "2026-10-31"  #@param {type:"date"}

#@markdown ### Products
PRODUCTS = "SST + LST"  #@param ["SST + LST", "SST", "LST"]
SST_PASS = "both"  #@param ["both", "D", "N"]
LST_PASS = "D"  #@param ["D", "N"]
LST_PLATFORM = "both"  #@param ["both", "terra", "aqua"]
#@markdown Optional Planetary Computer key (raises rate limits; anonymous access works without it).
PC_SDK_SUBSCRIPTION_KEY = ""  #@param {type:"string"}

#@markdown ### Climatology baseline (inclusive years)
BASELINE_FIRST_YEAR = 2018  #@param {type:"integer"}
BASELINE_LAST_YEAR = 2025  #@param {type:"integer"}

#@markdown ### Outputs
MAKE_PNG = True  #@param {type:"boolean"}
MAKE_GIF = True  #@param {type:"boolean"}
GIF_FPS = 4  #@param {type:"number"}
GEE_PROJECT = "ee-chriscandido93"  #@param {type:"string"}

import pathlib

from pipeline import config, gee, modis, mpc, periods, plot, runner, timeseries

REQUEST = periods.time_range(START_DATE, END_DATE)
OUTPUT_DIR = pathlib.Path(PIPELINE_DIR) / "outputs"
mpc.set_cache_dir(pathlib.Path(PIPELINE_DIR) / "cache")
if PC_SDK_SUBSCRIPTION_KEY:
    os.environ["PC_SDK_SUBSCRIPTION_KEY"] = PC_SDK_SUBSCRIPTION_KEY
SELECTED = ["SST", "LST"] if PRODUCTS == "SST + LST" else [PRODUCTS]
PASSES = {"SST": SST_PASS, "LST": LST_PASS}
BASELINE = (BASELINE_FIRST_YEAR, BASELINE_LAST_YEAR)

print(f"Range: {REQUEST.start} -> {REQUEST.last} ({REQUEST.length_days} days)")
print("Months:", ", ".join(f"{c.start:%Y/%m} ({c.start:%d}-{c.last:%d})" for c in periods.split_by_month(REQUEST)))
print("Products:", SELECTED, "| passes:", {p: PASSES[p] for p in SELECTED}, "| LST platform:", LST_PLATFORM,
      "| baseline:", f"{BASELINE[0]}-{BASELINE[1]}")

In [ ]:
#@title 3. Authenticate and initialise Earth Engine (SST)
if "SST" in SELECTED:
    gee.initialize(GEE_PROJECT)
    print("Earth Engine ready, project:", GEE_PROJECT)
else:
    print("LST only: Earth Engine not needed.")

In [ ]:
#@title 4. Run the pipeline (only missing outputs are processed)
summaries = {}
for name in SELECTED:
    summaries[name] = runner.run_range(
        config.PRODUCTS[name], REQUEST.start, REQUEST.last, sat_pass=PASSES[name], baseline=BASELINE,
        platform=LST_PLATFORM, out=OUTPUT_DIR, png=MAKE_PNG, gif=MAKE_GIF, gif_fps=GIF_FPS)

print()
for name, s in summaries.items():
    pending = f"{s['pending'].start} -> {s['pending'].last}" if s["pending"] else "none"
    print(f"{name}: newest data {s['latest']} | written {len(s['written'])} file(s) | "
          f"already there {len(s['skipped'])} | no images {len(s['no_data'])} | pending {pending}")

## Visualisation

In [ ]:
#@title 5. Monthly maps for the range
import datetime as dt
import json
import matplotlib.pyplot as plt

def monthly_outputs(name):
    product = config.PRODUCTS[name]
    found = []
    for chunk in periods.split_by_month(REQUEST):
        for kind in runner.KINDS:
            tif = runner.monthly_tif(OUTPUT_DIR, product, chunk.start, kind)
            if tif.exists():
                meta = json.loads(tif.with_suffix(".json").read_text())
                span = periods.Period(dt.date.fromisoformat(meta["period_start"]),
                                      dt.date.fromisoformat(meta["period_end_exclusive"]))
                found.append({"product": name, "kind": kind, "period": span, "tif": tif})
    return found

MONTHLY = [r for name in SELECTED for r in monthly_outputs(name)]
fig = plot.results_panel(MONTHLY)
if fig is None:
    print("No monthly outputs in this range yet.")
else:
    plt.show()

In [ ]:
#@title 6. GIFs of the range
from IPython.display import Image, display

for name, s in summaries.items():
    for gif in s["gifs"]:
        print(gif)
        display(Image(filename=str(gif)))
if not any(s["gifs"] for s in summaries.values()):
    print("No GIF made (MAKE_GIF is off, or no outputs in the range).")

In [ ]:
#@title 7. Interactive map of the monthly outputs (zoom, toggle layers)
import geemap
import ipyleaflet
from google.colab import output

output.enable_custom_widget_manager()

m = geemap.Map(center=[15, 125], zoom=5, ee_initialize=False)
m.add_basemap("Esri.WorldGrayCanvas")
colorbars = {}
for r in MONTHLY:
    product = config.PRODUCTS[r["product"]]
    if r["kind"] == "anomaly":
        vmin, vmax, _ = config.ANOMALY_VIS
        vis, label = {"min": vmin, "max": vmax, "palette": config.ANOMALY_PALETTE}, f"{product.name} anomaly (°C)"
    else:
        vmin, vmax = product.temp_vis
        vis, label = {"min": vmin, "max": vmax, "palette": config.TEMP_PALETTE}, f"{product.name} (°C)"
    url, bounds = plot.overlay_png(r["tif"], r["kind"], product)
    m.add_layer(ipyleaflet.ImageOverlay(url=url, bounds=bounds, name=f"{product.name} {r['kind']} {r['period'].label}"))
    colorbars.setdefault(label, vis)

m.add_layer(ipyleaflet.Polygon(locations=[(lat, lon) for lon, lat in config.PAR_COORDS], color="black",
                               fill=False, weight=2, name="PAR boundary"))
for label, vis in colorbars.items():
    m.add_colorbar(vis, label=label)
m

In [ ]:
#@title 8. PAR-mean monthly time series over the whole record (anomaly vs the baseline)
TS_PRODUCT = "SST"  #@param ["SST", "LST"]
#@markdown LST reads every 8-day composite since 2000 on a coarse grid: the first run takes a while
#@markdown (finished years are cached in `SDMAD_Products/cache/`, so later runs only read the current year).
import csv

product = config.PRODUCTS[TS_PRODUCT]
if TS_PRODUCT == "SST":
    if "SST" not in SELECTED:
        gee.initialize(GEE_PROJECT)
    coll = gee.collection(product, PASSES["SST"])
    first, latest = gee.date_range(coll)
    print(f"Computing monthly PAR means {first} -> {latest} (this can take a minute)...")
    series = timeseries.monthly_area_mean(coll, first, latest)
    rows = timeseries.monthly_anomalies(series, *BASELINE)
    tag = PASSES["SST"]
else:
    plats = modis.platforms_for(LST_PLATFORM)
    latest = modis.latest_date(plats)
    first = min(modis.RECORD_START[p] for p in plats)
    print(f"Computing monthly PAR-land means {first} -> {latest} for {', '.join(plats)}...")
    per_platform = modis.monthly_series(first, latest, PASSES["LST"], LST_PLATFORM)
    rows = timeseries.combine_platforms(
        {p: timeseries.monthly_anomalies(s, *BASELINE) for p, s in per_platform.items()})
    tag = f"{PASSES['LST']}_{LST_PLATFORM}"

ts_dir = OUTPUT_DIR / product.name / "timeseries"
ts_dir.mkdir(parents=True, exist_ok=True)
stem = ts_dir / f"{product.name}_PAR_monthly_{tag}_baseline{BASELINE[0]}-{BASELINE[1]}"
with open(f"{stem}.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["month", "mean_degC", "climatology_degC", "anomaly_degC"])
    for d, v, c, a in rows:
        w.writerow([d.strftime("%Y-%m")] + ["" if x is None else round(x, 3) for x in (v, c, a)])

fig = plot.timeseries_figure(rows, product)
fig.savefig(f"{stem}.png", dpi=150)
plt.show()
print("Saved", f"{stem}.csv", "and", f"{stem}.png")